# Local Density Detection: Local Outlier Factor for Station-Level Reporting Anomalies

**Notebook 07 of the spatiotemporal air quality anomaly detection pipeline**

---

This notebook applies the Local Outlier Factor (Breunig et al., 2000) to the shared station
feature profiles, measuring how a station's local density in feature space compares to the
density of its own neighbourhood — a station whose neighbourhood is locally sparse relative
to the neighbourhoods around it is a candidate anomaly, even if it would not stand out
against the dataset's global density.

| | |
|---|---|
| **Input** | `station_features.csv`, `config/params.yml` |
| **Output** | `station_suspicion_lof.csv` |
| **Downstream** | the consensus notebook (`consensus.ipynb`) — cross-method consensus |


## Contents

0. **Method Context**
   - 0.1 Objective
   - 0.2 Approach and Principles
   - 0.3 Inputs and Outputs
1. **Environment and Data**
   - 1.1 Configuration
   - 1.2 Input Loading and Validation
2. **Feature Profile Overview**
   - 2.1 Feature Contract
   - 2.2 Feature Completeness and Coverage
3. **Local Outlier Factor Estimation**
   - 3.1 Minimum Stations for Estimation
   - 3.2 Per-Country Model Fitting
4. **Suspicion Scoring and Flagging**
   - 4.1 Score Standardisation
   - 4.2 Flagging
5. **Model Diagnostics**
   - 5.1 Score Distributions by Country
   - 5.2 Profiles of Flagged and Unflagged Stations
   - 5.3 Sensitivity to the Neighbourhood Size
6. **Results and Handoff**
   - 6.1 Flagged Stations
   - 6.2 Consensus-Ready Output
   - 6.3 Output Validation
7. **Findings and Limitations**

---

Terminology

- **Local reachability density** is the inverse of the average distance from a point to its
  k nearest neighbours in feature space, adjusted so that dense regions yield a high value.
- The **Local Outlier Factor** is the ratio of a point's local reachability density to the
  average local reachability density of its neighbours; a value near one indicates density
  comparable to the neighbourhood, while a value well above one indicates a point in a
  sparser region than its neighbours occupy.
- A **detection feature** enters the model; a **diagnostic attribute** is carried alongside
  results but withheld from training, as established in Notebook 02.


## 0. Method Context

### 0.1 Objective

Isolation Forest (Notebook 04) and the autoencoder (Notebook 11) each judge a station
against a summary of the *entire* national network's feature structure — a random
partitioning process fitted to all stations, or a compressed representation trained to
reconstruct all of them. The Local Outlier Factor asks a more local question: relative to
the handful of stations nearest to it in feature space, is this station's neighbourhood
itself sparse?

This distinction matters because a network can contain more than one legitimate mode of
normal behaviour — a cluster of stations in a clean, low-variability region and a separate
cluster in a polluted, high-variability one, say — without either cluster being anomalous
relative to the whole dataset. A global method may judge every station against a single
network-wide standard; the Local Outlier Factor instead asks whether a station's *own local
neighbourhood* is as dense as the neighbourhoods elsewhere in the network, which can surface
a station straddling two modes or sitting in a sparse transition region that a global
standard would not flag.

Two objectives follow:

1. **Detect stations whose local neighbourhood in feature space is sparse relative to their
   network's other local neighbourhoods.** <br>This is a comparison of densities, not of
   values directly, and can catch a station a global method would consider unremarkable
   because it sits in an under-populated region of an otherwise well-populated feature
   space.</br>

2. **Contribute a fifth structurally independent signal to the consensus.** <br>Local
   density comparison is a distinct mechanism from isolation by partitioning, from
   reconstruction under compression, and from the global density measure DBSCAN's `eps`
   applies uniformly across a country. Agreement across this many independent mechanisms is
   stronger evidence than agreement across fewer.</br>


### 0.2 Approach and Principles

Four decisions govern the implementation.

1. **The feature table is shared, not recomputed.** <br>This notebook reads
   `station_features.csv`, built once in Notebook 02 and already consumed by Notebook 04,
   Notebook 10, and Notebook 11. An earlier version of this notebook recomputed its own
   features locally, using the mean and standard deviation rather than the median and
   interquartile range the rest of this pipeline adopts for skewed concentration data — that
   inconsistency is resolved by reading the shared table instead.</br>

2. **Per-country fitting.** <br>A separate LOF estimation is fitted for each national
   network, for the same reason given throughout this pipeline: pooling networks with
   different concentration regimes would let a station's country of origin dominate its
   local density comparison.</br>

3. **The suspicion score is standardised, not min-max scaled.** <br>An earlier version of
   this notebook rescaled the raw LOF score to a 0–1 range per country. Min-max scaling is
   sensitive to a single extreme value — one station with a very high score compresses every
   other station's rescaled position toward zero — and is inconsistent with the
   standardised, higher-means-more-suspicious convention every other method in this pipeline
   uses. This notebook instead standardises the LOF score to a z-score within its own
   country, consistent with Notebook 03 and Notebook 05.</br>

4. **The assumed anomaly proportion is shared with Isolation Forest.** <br>The contamination
   parameter LOF requires is read from the same configuration value Isolation Forest uses
   (Notebook 04), since contamination represents an assumption about the dataset — the
   expected share of anomalous stations in a network — rather than a property specific to
   either detection mechanism. Reusing a validated dataset-level assumption is preferred over
   introducing a second, independently unvalidated one.</br>


### 0.3 Inputs and Outputs

| Direction | Artifact | Contents |
|---|---|---|
| **In** | `station_features.csv` | Shared station-level feature profiles (Notebook 02, Section 9) |
| **In** | `config/params.yml` | Feature contract, contamination, LOF neighbourhood size |
| **Out** | `station_suspicion_lof.csv` | Per-station LOF score, suspicion score, and flag |

The output schema matches the other detection notebooks — `location_id`, `country`,
`suspicion_score`, `flagged` — so that the consensus notebook (`consensus.ipynb`) can combine methods without per-method
special handling.

This notebook reads no output from any other detection notebook. It shares an input with
Notebook 04, Notebook 10, and Notebook 11, not a verdict, and reaches its own conclusion from
that input by a mechanism none of those three use.


## 1. Environment and Data

### 1.1 Configuration

The feature contract is read from the same declaration Notebook 04, Notebook 10, and
Notebook 11 use. The contamination rate is read from the same configuration value Notebook 04
validated, rather than declared independently.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_style import table_style

# ── Configuration ────────────────────────────────────────────────────
CONFIG_PATH = Path("../config/params.yml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

REQUIRED_SECTIONS = ["meta", "station_features", "lof", "tsad", "paths", "colors_map"]
_missing = [s for s in REQUIRED_SECTIONS if s not in CONFIG]
if _missing:
    raise KeyError(
        f"Missing configuration section(s): {_missing}. 'station_features' is written by "
        f"Notebook 02 (Section 9); run it to completion first if that section is absent."
    )

PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
FIGURE_DIR    = PROCESSED_DIR.parent.parent / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_COLOURS = CONFIG["colors_map"]
COUNTRY_ORDER   = ["China", "Germany", "India", "USA"]
RANDOM_SEED     = CONFIG["meta"]["random_seed"]

DETECTION_FEATURES    = CONFIG["station_features"]["detection"]
DIAGNOSTIC_ATTRIBUTES = CONFIG["station_features"]["diagnostic"]

N_NEIGHBORS   = CONFIG["lof"]["n_neighbors"]
CONTAMINATION = CONFIG["tsad"]["isolation_forest_contamination"]

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)

parameters = pd.DataFrame([
    ("n_neighbors", N_NEIGHBORS, "Breunig et al. (2000) convention"),
    ("contamination", CONTAMINATION, "Shared with Notebook 04 (validated in NB02 §7.5)"),
    ("detection features", len(DETECTION_FEATURES), "Declared in params.yml (NB02 §9.1)"),
], columns=["Parameter", "Value", "Role"]).set_index("Parameter")

display(table_style(parameters))


<u>Interpretation</u>

The configuration loads, and both parameters this method depends on beyond its own
neighbourhood size are reused from where they were already validated: the feature contract
from Notebook 02, the contamination assumption from Notebook 04. Neither is re-derived or
re-validated here.


### 1.2 Input Loading and Validation

The shared feature table is the sole input, validated against the declared contract before
use.


In [ ]:
FEATURES_PATH = PROCESSED_DIR / "station_features.csv"
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"{FEATURES_PATH} not found. Run 02_exploratory_data_analysis.ipynb (Section 9) "
        f"to completion first; it is the sole source of station-level features."
    )

profiles = pd.read_csv(FEATURES_PATH)

REQUIRED_COLUMNS = {"location_id", "country", "assessed"} | set(DETECTION_FEATURES) | set(DIAGNOSTIC_ATTRIBUTES)
_absent = REQUIRED_COLUMNS - set(profiles.columns)
if _absent:
    raise KeyError(f"Fields declared in params.yml but absent from {FEATURES_PATH.name}: {_absent}")

loaded = pd.DataFrame([
    ("Feature file",      FEATURES_PATH.name),
    ("Stations profiled", f"{len(profiles):,}"),
    ("Countries",         ", ".join(sorted(profiles["country"].dropna().unique()))),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(loaded))


<u>Interpretation</u>

The same feature table Notebook 04, Notebook 10, and Notebook 11 consume loads here
unchanged. This is the resolution of the inconsistency noted in Section 0.2: this method now
reasons over the same median-and-interquartile-range feature definition every other
multivariate detector in this pipeline uses, rather than a locally recomputed
mean-and-standard-deviation alternative.


## 2. Feature Profile Overview

Feature construction is performed once in Notebook 02 (Section 9) and shared across every
multivariate detector. This section reviews the profile this notebook received, rather than
repeating its construction.


### 2.1 Feature Contract



In [ ]:
feature_registry = pd.DataFrame(
    [(f, "detection", "trained") for f in DETECTION_FEATURES] +
    [(a, "diagnostic", "withheld") for a in DIAGNOSTIC_ATTRIBUTES],
    columns=["Name", "Group", "Role in this model"]
).set_index("Name")

display(table_style(feature_registry))


<u>Interpretation</u>

Six features train the model and three attributes are carried alongside it, matching the
contract declared in `params.yml` — identical to what Notebook 04, Notebook 10, and Notebook
13 each read.


### 2.2 Feature Completeness and Coverage



In [ ]:
all_feature_cols = DETECTION_FEATURES + DIAGNOSTIC_ATTRIBUTES

completeness = pd.DataFrame({
    "missing_stations": profiles[all_feature_cols].isna().sum(),
    "missing_%": (profiles[all_feature_cols].isna().mean() * 100).round(2),
})
completeness["group"] = ["detection" if f in DETECTION_FEATURES else "diagnostic"
                         for f in completeness.index]
completeness = completeness[["group", "missing_stations", "missing_%"]]

coverage_by_country = (profiles.groupby("country")
                       .agg(stations_profiled=("location_id", "nunique"))
                       .reindex([c for c in COUNTRY_ORDER if c in profiles["country"].values]))

display(table_style(completeness))
display(table_style(coverage_by_country))


<u>Interpretation</u>

Feature coverage and per-country station counts match what Notebook 04, Notebook 10, and
Notebook 11 each observed, as expected from a shared input. These counts are the input to the
minimum-station requirement for LOF estimation derived in Section 3.1.


## 3. Local Outlier Factor Estimation

With the shared feature profiles loaded, a separate LOF estimation is fitted per country.


### 3.1 Minimum Stations for Estimation

LOF requires at least `n_neighbors` other points to define a local neighbourhood at all, so
a country must have more stations than that floor for the method to be meaningful. The
requirement follows directly from the neighbourhood size itself, the same
requirement-follows-from-parameter reasoning applied to Isolation Forest's minimum station
count in Notebook 04.


In [ ]:
MIN_STATIONS_FOR_LOF = N_NEIGHBORS + 1

coverage = (profiles.groupby("country")
           .agg(stations=("location_id", "nunique"))
           .reindex([c for c in COUNTRY_ORDER if c in profiles["country"].values]))
coverage["meets_minimum"] = coverage["stations"] >= MIN_STATIONS_FOR_LOF
coverage["effective_k"] = coverage["stations"].apply(lambda n: min(N_NEIGHBORS, n - 1))

display(table_style(pd.DataFrame([
    ("Configured n_neighbors", N_NEIGHBORS),
    ("Derived minimum stations", MIN_STATIONS_FOR_LOF, ),
], columns=["Property", "Value"]).set_index("Property")))
display(table_style(coverage))


<u>Interpretation</u>

Any network below the minimum cannot support even one full-sized local neighbourhood and is
excluded from this method entirely rather than fitted with a degraded neighbourhood size
silently. Where a network's station count exceeds the minimum but is still modest, the
effective neighbourhood size actually used is capped at one fewer than the station count, so
that fitting never requests more neighbours than the network can supply.


### 3.2 Per-Country Model Fitting

Each country's stations are standardised and a separate `LocalOutlierFactor` model is fitted,
using the effective neighbourhood size established above.


In [ ]:
lof_frames = []
fit_log = []

for country in [c for c in COUNTRY_ORDER if c in profiles["country"].values]:
    subset = profiles[profiles["country"] == country].copy()

    if len(subset) < MIN_STATIONS_FOR_LOF:
        subset["lof_score"] = np.nan
        subset["assessed"] = False
        lof_frames.append(subset)
        fit_log.append({"country": country, "stations": len(subset), "fitted": False,
                        "note": f"below minimum of {MIN_STATIONS_FOR_LOF}"})
        continue

    X = subset[DETECTION_FEATURES].fillna(subset[DETECTION_FEATURES].median())
    X_scaled = StandardScaler().fit_transform(X)

    k = min(N_NEIGHBORS, len(subset) - 1)
    lof = LocalOutlierFactor(n_neighbors=k, contamination=CONTAMINATION)
    prediction = lof.fit_predict(X_scaled)

    subset["lof_score"] = -lof.negative_outlier_factor_
    subset["assessed"] = True
    lof_frames.append(subset)
    fit_log.append({"country": country, "stations": len(subset), "fitted": True,
                    "effective_k": k, "note": ""})

scored = pd.concat(lof_frames, ignore_index=True)

display(table_style(pd.DataFrame(fit_log).set_index("country")))


<u>Interpretation</u>

Every network meeting the minimum is fitted, using the neighbourhood size the coverage table
in Section 3.1 established as effective for that country. The raw LOF score is retained here
in its native scale; Section 4 is where it becomes the standardised suspicion score handed to
the consensus.


## 4. Suspicion Scoring and Flagging

The raw LOF score is converted to the pipeline's standard suspicion-score convention before
flagging.


### 4.1 Score Standardisation

The raw LOF score is standardised to a z-score within each country, consistent with the
per-country relative scoring used by Notebook 03 and Notebook 05. This replaces a min-max
rescaling used in an earlier version of this notebook: min-max compresses every station's
position toward zero in response to a single extreme value, since it rescales the entire
range between the observed minimum and maximum, whereas standardisation is comparatively
robust to one or two extreme scores and keeps this method's scores on the same
higher-means-more-suspicious footing every other method in this pipeline already uses.


In [ ]:
country_stats = (scored[scored["assessed"]]
                .groupby("country")["lof_score"]
                .agg(mean_score="mean", sd_score=lambda s: s.std(ddof=1))
                .reset_index())

scored = scored.merge(country_stats, on="country", how="left")
scored["suspicion_score"] = np.where(
    scored["assessed"] & (scored["sd_score"] > 0),
    (scored["lof_score"] - scored["mean_score"]) / scored["sd_score"],
    np.nan,
)
scored = scored.drop(columns=["mean_score", "sd_score"])

display(table_style(scored.nlargest(10, "suspicion_score")
                    [["location_id", "country", "lof_score", "suspicion_score"]]
                    .reset_index(drop=True).round(3)))


<u>Interpretation</u>

The highest-scoring stations sit in the sparsest local neighbourhoods relative to their own
country's typical local density, expressed on a scale directly comparable to the suspicion
scores of every other method in this pipeline, rather than a per-country 0–1 range whose
absolute position had no meaning across countries.


### 4.2 Flagging

A station is flagged if it is assessed and either scikit-learn's own contamination-based
decision boundary or the standardised score exceeds a conventional cutoff. The
`LocalOutlierFactor` prediction already applies the configured contamination rate; that
native flag is used directly rather than re-deriving a separate threshold on the
standardised score, since re-deriving one would only reintroduce the same contamination
assumption by another route.


In [ ]:
lof_predictions = {}
for country in [c for c in COUNTRY_ORDER if c in scored.loc[scored["assessed"], "country"].unique()]:
    subset = profiles[profiles["country"] == country]
    if len(subset) < MIN_STATIONS_FOR_LOF:
        continue
    X = subset[DETECTION_FEATURES].fillna(subset[DETECTION_FEATURES].median())
    X_scaled = StandardScaler().fit_transform(X)
    k = min(N_NEIGHBORS, len(subset) - 1)
    lof = LocalOutlierFactor(n_neighbors=k, contamination=CONTAMINATION)
    pred = lof.fit_predict(X_scaled)
    lof_predictions.update(dict(zip(subset["location_id"], pred == -1)))

scored["flagged"] = scored["location_id"].map(lof_predictions).fillna(False) & scored["assessed"]

flag_summary = pd.DataFrame([
    ("Stations assessed", int(scored["assessed"].sum())),
    ("Flagged as anomalous", int(scored["flagged"].sum())),
], columns=["Category", "Stations"]).set_index("Category")

display(table_style(flag_summary))


<u>Interpretation</u>

The flagged count follows the contamination rate applied per network, consistent with how
Isolation Forest's flags behave in Notebook 04 — the same caveat applies here: contamination
fixes how many stations are flagged, not how many genuinely are anomalous, so the continuous
suspicion score carries more information than the binary flag alone.


## 5. Model Diagnostics

### 5.1 Score Distributions by Country



In [ ]:
assessed = scored[scored["assessed"]]
countries_assessed = [c for c in COUNTRY_ORDER if c in assessed["country"].values]

fig, axes = plt.subplots(1, len(countries_assessed),
                         figsize=(4.2 * len(countries_assessed), 3.8), squeeze=False)
axes = axes.flatten()

for ax, country in zip(axes, countries_assessed):
    sub = assessed[assessed["country"] == country]
    ax.hist(sub["suspicion_score"], bins=30, alpha=0.85,
            color=COUNTRY_COLOURS.get(country, "#888888"))
    ax.set_title(f"{country} — {len(sub):,} stations")
    ax.set_xlabel("Suspicion score (standardised LOF)")

fig.suptitle("Suspicion score distribution by network", y=1.02)
plt.savefig(FIGURE_DIR / "10_score_distributions.png")
plt.show()


<u>Interpretation</u>

Most stations sit close to zero — local density comparable to their neighbourhood — with the
flagged stations in the right tail. The distributions are directly comparable across
countries now that scores are standardised rather than min-max rescaled, so a suspicion score
of 2 means the same thing — two standard deviations above that country's own mean — in every
panel.


### 5.2 Profiles of Flagged and Unflagged Stations

As with the equivalent sections in Notebook 04, Notebook 10, and Notebook 11, this is a
descriptive comparison, not a formal feature attribution.


In [ ]:
comparison_features = DETECTION_FEATURES + ["pct_very_low"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, feature in zip(axes, comparison_features):
    consistent = assessed.loc[~assessed["flagged"], feature].dropna()
    anomalous  = assessed.loc[assessed["flagged"],  feature].dropna()
    if consistent.empty or anomalous.empty:
        ax.axis("off")
        continue
    ax.hist(consistent, bins=30, alpha=0.6, density=True, color="#378ADD", label="Consistent")
    ax.hist(anomalous, bins=20, alpha=0.75, density=True, color="#E24B4A", label="Flagged")
    label = feature + ("  (withheld)" if feature in DIAGNOSTIC_ATTRIBUTES else "")
    ax.set_title(label, fontsize=9.5)
    ax.legend(fontsize=8)

for ax in axes[len(comparison_features):]:
    ax.axis("off")

fig.suptitle("Feature distributions: flagged against consistent stations", y=1.0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "10_feature_profiles.png")
plt.show()


<u>Interpretation</u>

A feature separating sharply between flagged and consistent stations describes what makes
the flagged group's local neighbourhood sparse. Compared against the equivalent figures in
Notebook 04, Notebook 10, and Notebook 11, a feature that separates under all four methods
points to a signature every mechanism independently recognises.


### 5.3 Sensitivity to the Neighbourhood Size

`n_neighbors` determines the scale at which local density is measured, and the flagged set's
dependence on it is tested directly, in the same spirit as the parameter-sensitivity checks
in Notebook 04 and Notebook 10.


In [ ]:
CANDIDATE_K = [10, 20, 30, 40]

sensitivity_rows = []
for country in countries_assessed:
    subset = profiles[profiles["country"] == country]
    if len(subset) < min(CANDIDATE_K) + 1:
        continue
    X = subset[DETECTION_FEATURES].fillna(subset[DETECTION_FEATURES].median())
    X_scaled = StandardScaler().fit_transform(X)

    flagged_sets = {}
    for k in CANDIDATE_K:
        k_eff = min(k, len(subset) - 1)
        lof = LocalOutlierFactor(n_neighbors=k_eff, contamination=CONTAMINATION)
        pred = lof.fit_predict(X_scaled)
        flagged_sets[k] = set(subset.loc[pred == -1, "location_id"])

    baseline_set = flagged_sets[N_NEIGHBORS] if N_NEIGHBORS in flagged_sets else flagged_sets[CANDIDATE_K[1]]
    row = {"country": country}
    for k in CANDIDATE_K:
        row[f"flagged @ k={k}"] = len(flagged_sets[k])
    always = set.intersection(*flagged_sets.values())
    ever = set.union(*flagged_sets.values())
    row["stability_%"] = round(len(always) / max(len(ever), 1) * 100, 1)
    sensitivity_rows.append(row)

display(table_style(pd.DataFrame(sensitivity_rows).set_index("country")))


<u>Interpretation</u>

A high stability figure means the flagged set changes little across a reasonable range of
neighbourhood sizes, and the specific value of 20 is not doing unseen work in determining the
result. A country with low stability here indicates that this method's verdict for that
network is comparatively sensitive to a parameter choice, and its flags should be weighted
accordingly in the consensus.


## 6. Results and Handoff

### 6.1 Flagged Stations



In [ ]:
ranked = (scored[scored["flagged"]]
          .sort_values("suspicion_score", ascending=False)
          [["location_id", "country", "lof_score", "suspicion_score",
            "median_value", "iqr_value", "pct_missing"]]
          .reset_index(drop=True)
          .round(3))

display(table_style(ranked.head(15)))


<u>Interpretation</u>

These stations sit in the sparsest local neighbourhoods relative to their own network. As
with every method in this pipeline, they are candidates rather than conclusions; their
standing is settled in the consensus notebook (`consensus.ipynb`) alongside the spatial baseline, Isolation Forest, DBSCAN,
Benford, and the autoencoder.


### 6.2 Consensus-Ready Output



In [ ]:
output = scored[[
    "location_id", "country",
    *DETECTION_FEATURES, *DIAGNOSTIC_ATTRIBUTES,
    "lof_score", "suspicion_score", "assessed", "flagged",
]].copy()

output["method"] = "lof"
output = output.sort_values("suspicion_score", ascending=False, na_position="last")

OUTPUT_PATH = PROCESSED_DIR / "station_suspicion_lof.csv"
output.to_csv(OUTPUT_PATH, index=False)

manifest = pd.DataFrame([
    ("Output file", OUTPUT_PATH.name),
    ("Stations written", f"{len(output):,}"),
    ("Flagged", f"{int(output['flagged'].sum()):,}"),
    ("Assessed", f"{int(output['assessed'].sum()):,}"),
    ("Consensus schema", "location_id, country, suspicion_score, flagged"),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(manifest))


<u>Interpretation</u>

The result is persisted in the schema shared by every detection notebook, with the
standardised score — not a per-country 0–1 rescaling — as the suspicion score the consensus
receives.


### 6.3 Output Validation



In [ ]:
reloaded = pd.read_csv(OUTPUT_PATH)

checks = pd.DataFrame([
    ("Stations written", len(output), len(reloaded), len(output) == len(reloaded)),
    ("Unique station IDs", output["location_id"].nunique(), reloaded["location_id"].nunique(),
     output["location_id"].nunique() == reloaded["location_id"].nunique()),
    ("Flagged count", int(output["flagged"].sum()), int(reloaded["flagged"].sum()),
     int(output["flagged"].sum()) == int(reloaded["flagged"].sum())),
    ("Consensus columns present", "yes",
     "yes" if {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns) else "no",
     {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns)),
], columns=["Check", "Computed", "Reloaded", "Pass"]).set_index("Check")

display(table_style(checks))

assert len(output) == len(reloaded), "Row count changed on write."
assert output["location_id"].duplicated().sum() == 0, "Duplicate station in output."
assert {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns), \
    "Consensus schema incomplete; the consensus notebook (`consensus.ipynb`) would fail on this file."


<u>Interpretation</u>

The written file reconciles with the computed result and carries the consensus schema
intact.


## 7. Findings and Limitations

### Findings

1. **A fifth mechanism, structurally independent of the other four multivariate and
consensus-relevant methods.** <br>Local density comparison detects stations that a global
standard would consider unremarkable if their own neighbourhood in feature space is sparse
relative to the network's other local neighbourhoods.</br>

2. **Two inconsistencies with pipeline convention were corrected in this restructuring.**
<br>The earlier version of this notebook recomputed its own mean/standard-deviation
features rather than reading the shared, robust `station_features.csv`, and rescaled its
suspicion score by min-max rather than standardising it. Both are now aligned with the rest
of the pipeline.</br>

3. **Contamination is shared, not independently assumed.** <br>Reusing the value Notebook 04
validated treats contamination as what it actually represents — an assumption about the
dataset, not a property specific to either detection mechanism — rather than introducing a
second unvalidated instance of the same assumption.</br>

### Limitations

1. **`n_neighbors` sets the scale of "local".** <br>A neighbourhood size much smaller than
the configured value would detect finer-grained local structure; much larger would approach
the global comparison Isolation Forest and the autoencoder already provide. Section 5.3
quantifies how much the flagged set actually depends on this choice.</br>

2. **No formal feature attribution.** <br>As with the other multivariate methods in this
pipeline, LOF does not report which feature drove an individual station's density
comparison; Section 5.2 compares distributions between groups without attributing an
individual verdict to a specific feature.</br>

3. **Sparse networks cannot be assessed.** <br>A country with fewer stations than
`n_neighbors + 1` cannot support even one full local neighbourhood and is excluded from this
method entirely.</br>

4. **Shared dependence on the feature table's construction choices.** <br>Any limitation
attached to `station_features.csv` itself applies here identically to how it applies in
Notebook 04, Notebook 10, and Notebook 11, since all four notebooks consume the same
table.</br>

---

### Output

| File | Contents |
|---|---|
| `station_suspicion_lof.csv` | Per-station LOF score, suspicion score, flag, and feature profile |

**Next:** `08_frozen_values.ipynb` — a temporal-pattern detector targeting stuck or fabricated
sensor readings, the final method before the cross-method consensus.

---

### References

Breunig, M. M., Kriegel, H.-P., Ng, R. T., & Sander, J. (2000). LOF: Identifying
density-based local outliers. *ACM SIGMOD Record*, 29(2), 93–104.
